# Caculate F1 score

In [1]:
from sklearn.metrics import f1_score, accuracy_score
import pickle
import numpy as np

def evaluate_by_batch(loaded_results, batch_size=None):
    attributes = list(loaded_results['predictions'].keys())
    results_summary = {}

    for attr in attributes:
        preds = loaded_results['predictions'][attr].squeeze()
        targets = loaded_results['targets'][attr].squeeze()

        binary_preds = (preds >= 0.5).astype(int)
        binary_targets = targets.astype(int)

        if batch_size is None:
            f1 = f1_score(binary_targets, binary_preds)
            acc = accuracy_score(binary_targets, binary_preds)
            results_summary[attr] = {
                'F1-score': round(f1, 4),
                'Accuracy': round(acc, 4)
            }
        else:
            f1_list, acc_list = [], []
            for i in range(0, len(binary_targets), batch_size):
                batch_preds = binary_preds[i:i+batch_size]
                batch_targets = binary_targets[i:i+batch_size]

                if len(np.unique(batch_targets)) > 1:
                    f1 = f1_score(batch_targets, batch_preds)
                else:
                    f1 = 1.0 if (batch_targets == batch_preds).all() else 0.0

                acc = accuracy_score(batch_targets, batch_preds)
                f1_list.append(f1)
                acc_list.append(acc)

            f1_mean, f1_std = np.mean(f1_list), np.std(f1_list)
            acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)

            results_summary[attr] = {
                'F1-score': round(f1_mean, 4),
                'F1-std': round(f1_std, 4),
                'Accuracy': round(acc_mean, 4),
                'Acc-std': round(acc_std, 4)
            }

    return results_summary

# === 加载结果文件 ===

do_file = 'Young'
with open("./saved/celeA_complex/{}/final_results.pkl".format(do_file), "rb") as f:
    loaded_results = pickle.load(f)

# === 设置 batch_size（None 为全数据评估）===
batch_size = 256  # 可设为 None

# === 评估 ===
results = evaluate_by_batch(loaded_results, batch_size=batch_size)

# === 打印结果 ===
print("do({}) with".format(do_file) + (f" batch_size={batch_size}" if batch_size else " full-dataset evaluation"))
for attr, scores in results.items():
    if batch_size is None:
        print(f"{attr}: F1 = {scores['F1-score']}, Acc = {scores['Accuracy']}")
    else:
        print(f"{attr}: F1 = {scores['F1-score']} ± {scores['F1-std']}, Acc = {scores['Accuracy']} ± {scores['Acc-std']}")

FileNotFoundError: [Errno 2] No such file or directory: './saved/celeA_complex/Young/final_results.pkl'

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
import pickle
import numpy as np
import os

def evaluate_by_batch(loaded_results, batch_size=None):
    attributes = list(loaded_results['predictions'].keys())
    results_summary = {}
    #print('Sample number',loaded_results['predictions'][attributes[0]].squeeze().shape)
    for attr in attributes:
        preds = loaded_results['predictions'][attr].squeeze()
        targets = loaded_results['targets'][attr].squeeze()

        binary_preds = (preds >= 0.5).astype(int)
        binary_targets = targets.astype(int)

        if batch_size is None:
            f1 = f1_score(binary_targets, binary_preds)
            acc = accuracy_score(binary_targets, binary_preds)
            results_summary[attr] = {
                'F1-score': round(f1, 3),
                'Accuracy': round(acc, 3)
            }
        else:
            f1_list, acc_list = [], []
            for i in range(0, len(binary_targets), batch_size):
                batch_preds = binary_preds[i:i+batch_size]
                batch_targets = binary_targets[i:i+batch_size]

                if len(np.unique(batch_targets)) > 1:
                    f1 = f1_score(batch_targets, batch_preds)
                else:
                    f1 = 1.0 if (batch_targets == batch_preds).all() else 0.0

                acc = accuracy_score(batch_targets, batch_preds)
                f1_list.append(f1)
                acc_list.append(acc)

            f1_mean, f1_std = np.mean(f1_list), np.std(f1_list)
            acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)

            results_summary[attr] = {
                'F1-score': round(f1_mean, 3),
                'F1-std': round(f1_std, 2),
                'Accuracy': round(acc_mean, 3),
                'Acc-std': round(acc_std, 2)
            }

    return results_summary


# === 设置评估目标属性 ===
target_attr = "Bald"
batch_size = None

# === 根目录 ===
root_dir = "../saved_ablation/celeA_complex/effectiveness_step100.0_scale2.5_NTITrue"

# === 遍历所有 do(attr) 文件夹 ===
do_folders = [name for name in ['Young','Male','No_Beard','Bald'] if os.path.isdir(os.path.join(root_dir, name))]


print(f"Comparing '{batch_size}' F1-score on attribute '{target_attr}' under different do(.) conditions:")

for do_attr in do_folders:
    result_path = os.path.join(root_dir, do_attr, "final_results.pkl")
    if not os.path.exists(result_path):
        continue  # skip missing

    with open(result_path, "rb") as f:
        loaded_results = pickle.load(f)

    results = evaluate_by_batch(loaded_results, batch_size=batch_size)

    if target_attr not in results:
        continue  # skip if target attr not in result

    scores = results[target_attr]
    if batch_size is None:
        print(f"do({do_attr})\t{target_attr}: F1 = {scores['F1-score']}, Acc = {scores['Accuracy']}")
    else:
        print(f"do({do_attr})\t{target_attr}: F1 = {scores['F1-score']} ± {scores['F1-std']}, Acc = {scores['Accuracy']} ± {scores['Acc-std']}")


Comparing 'None' F1-score on attribute 'Bald' under different do(.) conditions:
do(Young)	Bald: F1 = 0.186, Acc = 0.772
do(Male)	Bald: F1 = 0.062, Acc = 0.94
do(No_Beard)	Bald: F1 = 0.4, Acc = 0.988
do(Bald)	Bald: F1 = 0.082, Acc = 0.058


In [51]:
from sklearn.metrics import f1_score, accuracy_score
import pickle
import numpy as np
import os

def evaluate_by_batch(loaded_results, batch_size=None):
    attributes = list(loaded_results['predictions'].keys())
    results_summary = {}
    #print('Shape',loaded_results['predictions'][attributes[0]].squeeze().shape)
    for attr in attributes:
        preds = loaded_results['predictions'][attr].squeeze()[:400]
        targets = loaded_results['targets'][attr].squeeze()[:400]

        binary_preds = (preds >= 0.5).astype(int)
        binary_targets = targets.astype(int)

        if batch_size is None:
            f1 = f1_score(binary_targets, binary_preds)
            acc = accuracy_score(binary_targets, binary_preds)
            if len(np.unique(binary_targets)) == 1:
                f1 = 1.0 if (binary_targets == binary_preds).all() else 0.0
            results_summary[attr] = {
                'F1-score': round(f1, 3),
                'Accuracy': round(acc, 3)
            }
        else:
            f1_list, acc_list = [], []
            for i in range(0, len(binary_targets), batch_size):
                batch_preds = binary_preds[i:i+batch_size]
                batch_targets = binary_targets[i:i+batch_size]

                if len(np.unique(batch_targets)) > 1:
                    f1 = f1_score(batch_targets, batch_preds)
                else:
                    f1 = 1.0 if (batch_targets == batch_preds).all() else 0.0

                acc = accuracy_score(batch_targets, batch_preds)
                f1_list.append(f1)
                acc_list.append(acc)

            f1_mean, f1_std = np.mean(f1_list), np.std(f1_list)
            acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)

            results_summary[attr] = {
                'F1-score': round(f1_mean, 3),
                'F1-std': round(f1_std, 2),
                'Accuracy': round(acc_mean, 3),
                'Acc-std': round(acc_std, 2)
            }

    return results_summary


# === 设置评估目标属性 ===
target_attr = "No_Beard"
batch_size = None

# === 根目录 ===
root_dir = "../saved_ablation/celeA_complex/DSCM_effectiveness_step100.0_scale5.0_NTIFalse"

# === 遍历所有 do(attr) 文件夹 ===
do_folders = [name for name in ['Young','Male','No_Beard','Bald'] if os.path.isdir(os.path.join(root_dir, name))]

for target_attr in ['Young','Male','No_Beard','Bald']:
    print(f"Comparing '{batch_size}' F1-score on attribute '{target_attr}' under different do(.) conditions:")

    for do_attr in do_folders:
        result_path = os.path.join(root_dir, do_attr, "final_results.pkl")
        if not os.path.exists(result_path):
            continue  # skip missing

        with open(result_path, "rb") as f:
            loaded_results = pickle.load(f)

        results = evaluate_by_batch(loaded_results, batch_size=batch_size)

        if target_attr not in results:
            continue  # skip if target attr not in result

        scores = results[target_attr]
        if batch_size is None:
            print(f"do({do_attr})\t{target_attr}: F1 = {scores['F1-score']}, Acc = {scores['Accuracy']}")
        else:
            print(f"do({do_attr})\t{target_attr}: F1 = {scores['F1-score']} ± {scores['F1-std']}, Acc = {scores['Accuracy']} ± {scores['Acc-std']}")


Comparing 'None' F1-score on attribute 'Young' under different do(.) conditions:
do(Young)	Young: F1 = 0.634, Acc = 0.735
do(Male)	Young: F1 = 0.907, Acc = 0.86
do(No_Beard)	Young: F1 = 0.952, Acc = 0.922
do(Bald)	Young: F1 = 0.914, Acc = 0.878
Comparing 'None' F1-score on attribute 'Male' under different do(.) conditions:
do(Young)	Male: F1 = 0.997, Acc = 0.998
do(Male)	Male: F1 = 1.0, Acc = 1.0
do(No_Beard)	Male: F1 = 0.79, Acc = 0.79
do(Bald)	Male: F1 = 0.935, Acc = 0.945
Comparing 'None' F1-score on attribute 'No_Beard' under different do(.) conditions:
do(Young)	No_Beard: F1 = 1.0, Acc = 1.0
do(Male)	No_Beard: F1 = 0.98, Acc = 0.982
do(No_Beard)	No_Beard: F1 = 0.57, Acc = 0.792
do(Bald)	No_Beard: F1 = 0.978, Acc = 0.962
Comparing 'None' F1-score on attribute 'Bald' under different do(.) conditions:
do(Young)	Bald: F1 = 0.75, Acc = 0.995
do(Male)	Bald: F1 = 0.924, Acc = 0.958
do(No_Beard)	Bald: F1 = 0.667, Acc = 0.992
do(Bald)	Bald: F1 = 0.63, Acc = 0.465


In [53]:

for do_attr in do_folders:
    result_path = os.path.join(root_dir, do_attr, "final_results.pkl")
    if not os.path.exists(result_path):
        continue  # skip missing

    with open(result_path, "rb") as f:
        loaded_results = pickle.load(f)
    print('do {} lipips distance is {}'.format(do_attr,np.mean(loaded_results['lpips'])))

do Young lipips distance is 0.22728461027145386
do Male lipips distance is 0.3535422682762146
do No_Beard lipips distance is 0.21237201988697052
do Bald lipips distance is 0.338556706905365


In [29]:
#dict_keys(['predictions', 'targets'])
loaded_results.keys()
#dict_keys(['Young', 'Male', 'No_Beard', 'Bald'])
loaded_results['predictions'].keys()

dict_keys(['Young', 'Male', 'No_Beard', 'Bald'])

# FID and Minimality

In [1]:
import torch
minimality = torch.load("./saved/celeA_complex/FID/minimality.pt")
countefactual_images =  torch.load("./saved/celeA_complex/FID/counterfactual_tensors.pt")

/tmp/ipykernel_290807/4134258590.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  minimality = torch.load("./saved/celeA_complex/FID/minimality.pt")
/tmp/ipykernel_290807

In [6]:
minimality['factuals'][1].shape

AttributeError: 'tuple' object has no attribute 'shape'